In [ ]:
# @title SECUENCIAS SALUD
import os
import json
import unicodedata
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ==========================
# RUTAS
# ==========================
JSON_CANDIDATOS = [
    "/content/selecciones_salud_final.json",
    "/mnt/data/selecciones_salud_final.json",
]

# carpeta con todos los pictogramas
PICTOS_DIR = "/content/todos_pictogramas"

# ==========================
# PARÁMETROS OPCIONALES
# ==========================
MAX_ORACIONES_A_MOSTRAR = None   # p.ej. 20 para limitar; None para todas
EXT_PERMITIDAS = {".jpg", ".jpeg", ".png"}

#   "secuencia_final"  -> 1 pictograma por token (la selección final)
#   "elecciones"       -> todas las imágenes de 'elecciones' (para revisar candidatos)
MODO_IMAGENES = "secuencia_final"

# ==========================
# UTILIDADES
# ==========================
def normalizar_txt(s: str) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = s.replace(" ", "_")
    return s

def extraer_id(nombre: str):
    """
    Si nombre es '7106-dolor_de_cabeza.jpg' devuelve '7106'.
    Si no hay guion, devuelve None.
    """
    base = os.path.basename(nombre)
    if "-" in base:
        return base.split("-", 1)[0]
    return None

def quitar_prefijo_orden(nombre: str):
    """
    Quita prefijos tipo '01_', '002_' SOLO si están al inicio del nombre.
    Ej.: '01_dolor_de_cabeza.jpg' -> 'dolor_de_cabeza.jpg'
         '7156-dolor_de_cabeza.jpg' -> (sin cambios)
    """
    import re, os
    base = os.path.basename(nombre or "")
    return re.sub(r'^\d{1,4}_', '', base)


def resolver_ruta_picto(nombre_entrada: str,
                        index_por_nombre, index_por_stem, index_por_id):
    import unicodedata, os

    def sin_tildes(s: str) -> str:
        if not s: return s
        nf = unicodedata.normalize("NFD", s)
        return "".join(ch for ch in nf if unicodedata.category(ch) != "Mn")

    if not nombre_entrada:
        return None

    candidatos_nombre = [os.path.basename(nombre_entrada)]
    # Intenta también sin prefijo tipo '01_'
    nombre_sin_pref = quitar_prefijo_orden(nombre_entrada)
    if nombre_sin_pref != candidatos_nombre[0]:
        candidatos_nombre.append(nombre_sin_pref)

    for nombre in candidatos_nombre:
        base = os.path.basename(nombre)
        base_norm = normalizar_txt(base)
        stem_norm = normalizar_txt(os.path.splitext(base)[0])

        # 1) Búsqueda exacta por nombre+ext
        if base_norm in index_por_nombre:
            return index_por_nombre[base_norm]

        # 2) Probar otras extensiones si el stem coincide
        for ext in EXT_PERMITIDAS:
            cand = stem_norm + ext
            if cand in index_por_nombre:
                return index_por_nombre[cand]

        # 3) Por stem (sin extensión)
        if stem_norm in index_por_stem:
            return index_por_stem[stem_norm]

        # 3b) Fallback: sin tildes
        stem_noacc = sin_tildes(stem_norm)
        if stem_noacc != stem_norm:
            for ext in EXT_PERMITIDAS:
                cand = stem_noacc + ext
                if cand in index_por_nombre:
                    return index_por_nombre[cand]
            if stem_noacc in index_por_stem:
                return index_por_stem[stem_noacc]

        # 4) Por ID (parte antes del guion)
        _id = extraer_id(base)
        if _id and _id in index_por_id:
            return index_por_id[_id]

        # 5) Variantes guion/guion_bajo
        variantes = {
            base_norm.replace("-", "_"),
            base_norm.replace("_", "-"),
            stem_norm.replace("-", "_"),
            stem_norm.replace("_", "-"),
        }
        for var in variantes:
            # probar con extensión
            for ext in EXT_PERMITIDAS:
                cand = var.replace(ext, "") + ext
                if cand in index_por_nombre:
                    return index_por_nombre[cand]
            # probar sin extensión por stem
            if var in index_por_stem:
                return index_por_stem[var]

    return None

def elegir_json_path(candidatos):
    for p in candidatos:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "No se encontró el archivo JSON. "
        f"Probé: {', '.join(candidatos)}"
    )

# ==========================
# INDEXAR PICTOGRAMAS
# ==========================
print("Indexando pictogramas... (esto es rápido y solo se hace una vez)")

index_por_nombre = {}  # nombre exacto (normalizado) -> ruta
index_por_stem = {}    # nombre sin extensión (normalizado) -> ruta
index_por_id = {}      # ID numérica (texto) -> ruta (primera aparición)

total_archivos = 0
for root, _, files in os.walk(PICTOS_DIR):
    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext not in EXT_PERMITIDAS:
            continue
        total_archivos += 1
        ruta = os.path.join(root, fname)

        fname_norm = normalizar_txt(fname)
        stem_norm = normalizar_txt(os.path.splitext(fname)[0])

        index_por_nombre[fname_norm] = ruta
        if stem_norm not in index_por_stem:
            index_por_stem[stem_norm] = ruta

        _id = extraer_id(fname)
        if _id and _id.isdigit() and _id not in index_por_id:
            index_por_id[_id] = ruta

print(f"Archivos indexados: {total_archivos}")

# ==========================
# CARGAR JSON
# ==========================
JSON_PATH = elegir_json_path(JSON_CANDIDATOS)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} oraciones desde {JSON_PATH}.")

# Validar clave de imágenes elegida
if MODO_IMAGENES not in {"secuencia_final", "elecciones"}:
    raise ValueError("MODO_IMAGENES debe ser 'secuencia_final' o 'elecciones'.")

# ==========================
# MOSTRAR ORACIONES + PICTOGRAMAS
# ==========================
no_encontrados_global = []
mostradas = 0

for idx, item in enumerate(data, start=1):
    if MAX_ORACIONES_A_MOSTRAR is not None and mostradas >= MAX_ORACIONES_A_MOSTRAR:
        break

    oracion = item.get("oracion", "")
    lista_imgs = item.get(MODO_IMAGENES, []) or []
    # filtrar vacíos/null
    pictos = [p for p in lista_imgs if p]

    print(f"\n{idx}. {oracion}")
    if not pictos:
        print(f"   No hay imágenes en '{MODO_IMAGENES}' para esta oración.")
        continue

    imgs = []
    faltantes = []
    for fname in pictos:
        ruta = resolver_ruta_picto(fname, index_por_nombre, index_por_stem, index_por_id)
        if ruta and os.path.exists(ruta):
            try:
                img = mpimg.imread(ruta)
                imgs.append((os.path.basename(ruta), img))
            except Exception as e:
                print(f"   Error al leer {ruta}: {e}")
                faltantes.append(fname)
        else:
            faltantes.append(fname)

    if faltantes:
        for f in faltantes:
            print(f"   No se encontró la imagen: {f}")
        no_encontrados_global.extend(faltantes)

    if not imgs:
        print("   No se encontraron imágenes válidas para mostrar.")
        continue

    # Mostrar imágenes en fila
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]

    for ax, (fname_img, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        # Título debajo de cada imagen: parte después del '-' sin extensión
        titulo = fname_img
        if "-" in fname_img:
            titulo = fname_img.split("-", 1)[-1]
        titulo = titulo.rsplit(".", 1)[0].replace("_", " ")
        ax.set_title(titulo, fontsize=10)

    plt.suptitle(f"{oracion}", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

    mostradas += 1

# ==========================
# RESUMEN FINAL
# ==========================
no_encontrados_set = list(dict.fromkeys(no_encontrados_global))  # únicos, preservando orden
print("\n==========================")
print("RESUMEN DE IMÁGENES FALTANTES")
print("==========================")
print(f"Total faltantes (únicas): {len(no_encontrados_set)}")
if no_encontrados_set:
    print("Ejemplos:")
    for ej in no_encontrados_set[:20]:
        print("  -", ej)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title SECUENCIAS EDUCACION
import os
import json
import unicodedata
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ==========================
# RUTAS
# ==========================
JSON_CANDIDATOS = [
    "/content/selecciones_educacion_final.json",
    "/mnt/data/selecciones_educacion_final.json",
]

# carpeta con todos los pictogramas
PICTOS_DIR = "/content/todos_pictogramas"

# ==========================
# PARÁMETROS OPCIONALES
# ==========================
MAX_ORACIONES_A_MOSTRAR = None   # p.ej. 20 para limitar; None para todas
EXT_PERMITIDAS = {".jpg", ".jpeg", ".png"}

#   "secuencia_final"  -> 1 pictograma por token (la selección final)
#   "elecciones"       -> todas las imágenes de 'elecciones' (para revisar candidatos)
MODO_IMAGENES = "secuencia_final"

# ==========================
# UTILIDADES
# ==========================
def normalizar_txt(s: str) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = s.replace(" ", "_")
    return s

def extraer_id(nombre: str):
    """
    Si nombre es '7106-dolor_de_cabeza.jpg' devuelve '7106'.
    Si no hay guion, devuelve None.
    """
    base = os.path.basename(nombre)
    if "-" in base:
        return base.split("-", 1)[0]
    return None

def quitar_prefijo_orden(nombre: str):
    """
    Quita prefijos tipo '01_', '002_' SOLO si están al inicio del nombre.
    Ej.: '01_dolor_de_cabeza.jpg' -> 'dolor_de_cabeza.jpg'
         '7156-dolor_de_cabeza.jpg' -> (sin cambios)
    """
    import re, os
    base = os.path.basename(nombre or "")
    return re.sub(r'^\d{1,4}_', '', base)


def resolver_ruta_picto(nombre_entrada: str,
                        index_por_nombre, index_por_stem, index_por_id):
    import unicodedata, os

    def sin_tildes(s: str) -> str:
        if not s: return s
        nf = unicodedata.normalize("NFD", s)
        return "".join(ch for ch in nf if unicodedata.category(ch) != "Mn")

    if not nombre_entrada:
        return None

    candidatos_nombre = [os.path.basename(nombre_entrada)]
    # Intenta también sin prefijo tipo '01_'
    nombre_sin_pref = quitar_prefijo_orden(nombre_entrada)
    if nombre_sin_pref != candidatos_nombre[0]:
        candidatos_nombre.append(nombre_sin_pref)

    for nombre in candidatos_nombre:
        base = os.path.basename(nombre)
        base_norm = normalizar_txt(base)
        stem_norm = normalizar_txt(os.path.splitext(base)[0])

        # 1) Búsqueda exacta por nombre+ext
        if base_norm in index_por_nombre:
            return index_por_nombre[base_norm]

        # 2) Probar otras extensiones si el stem coincide
        for ext in EXT_PERMITIDAS:
            cand = stem_norm + ext
            if cand in index_por_nombre:
                return index_por_nombre[cand]

        # 3) Por stem (sin extensión)
        if stem_norm in index_por_stem:
            return index_por_stem[stem_norm]

        # 3b) Fallback: sin tildes
        stem_noacc = sin_tildes(stem_norm)
        if stem_noacc != stem_norm:
            for ext in EXT_PERMITIDAS:
                cand = stem_noacc + ext
                if cand in index_por_nombre:
                    return index_por_nombre[cand]
            if stem_noacc in index_por_stem:
                return index_por_stem[stem_noacc]

        # 4) Por ID (parte antes del guion)
        _id = extraer_id(base)
        if _id and _id in index_por_id:
            return index_por_id[_id]

        # 5) Variantes guion/guion_bajo
        variantes = {
            base_norm.replace("-", "_"),
            base_norm.replace("_", "-"),
            stem_norm.replace("-", "_"),
            stem_norm.replace("_", "-"),
        }
        for var in variantes:
            # probar con extensión
            for ext in EXT_PERMITIDAS:
                cand = var.replace(ext, "") + ext
                if cand in index_por_nombre:
                    return index_por_nombre[cand]
            # probar sin extensión por stem
            if var in index_por_stem:
                return index_por_stem[var]

    return None

def elegir_json_path(candidatos):
    for p in candidatos:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "No se encontró el archivo JSON. "
        f"Probé: {', '.join(candidatos)}"
    )

# ==========================
# INDEXAR PICTOGRAMAS
# ==========================
print("Indexando pictogramas... (esto es rápido y solo se hace una vez)")

index_por_nombre = {}  # nombre exacto (normalizado) -> ruta
index_por_stem = {}    # nombre sin extensión (normalizado) -> ruta
index_por_id = {}      # ID numérica (texto) -> ruta (primera aparición)

total_archivos = 0
for root, _, files in os.walk(PICTOS_DIR):
    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext not in EXT_PERMITIDAS:
            continue
        total_archivos += 1
        ruta = os.path.join(root, fname)

        fname_norm = normalizar_txt(fname)
        stem_norm = normalizar_txt(os.path.splitext(fname)[0])

        index_por_nombre[fname_norm] = ruta
        if stem_norm not in index_por_stem:
            index_por_stem[stem_norm] = ruta

        _id = extraer_id(fname)
        if _id and _id.isdigit() and _id not in index_por_id:
            index_por_id[_id] = ruta

print(f"Archivos indexados: {total_archivos}")

# ==========================
# CARGAR JSON
# ==========================
JSON_PATH = elegir_json_path(JSON_CANDIDATOS)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} oraciones desde {JSON_PATH}.")

# Validar clave de imágenes elegida
if MODO_IMAGENES not in {"secuencia_final", "elecciones"}:
    raise ValueError("MODO_IMAGENES debe ser 'secuencia_final' o 'elecciones'.")

# ==========================
# MOSTRAR ORACIONES + PICTOGRAMAS
# ==========================
no_encontrados_global = []
mostradas = 0

for idx, item in enumerate(data, start=1):
    if MAX_ORACIONES_A_MOSTRAR is not None and mostradas >= MAX_ORACIONES_A_MOSTRAR:
        break

    oracion = item.get("oracion", "")
    lista_imgs = item.get(MODO_IMAGENES, []) or []
    # filtrar vacíos/null
    pictos = [p for p in lista_imgs if p]

    print(f"\n{idx}. {oracion}")
    if not pictos:
        print(f"   No hay imágenes en '{MODO_IMAGENES}' para esta oración.")
        continue

    imgs = []
    faltantes = []
    for fname in pictos:
        ruta = resolver_ruta_picto(fname, index_por_nombre, index_por_stem, index_por_id)
        if ruta and os.path.exists(ruta):
            try:
                img = mpimg.imread(ruta)
                imgs.append((os.path.basename(ruta), img))
            except Exception as e:
                print(f"   Error al leer {ruta}: {e}")
                faltantes.append(fname)
        else:
            faltantes.append(fname)

    if faltantes:
        for f in faltantes:
            print(f"   No se encontró la imagen: {f}")
        no_encontrados_global.extend(faltantes)

    if not imgs:
        print("   No se encontraron imágenes válidas para mostrar.")
        continue

    # Mostrar imágenes en fila
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]

    for ax, (fname_img, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        # Título debajo de cada imagen: parte después del '-' sin extensión
        titulo = fname_img
        if "-" in fname_img:
            titulo = fname_img.split("-", 1)[-1]
        titulo = titulo.rsplit(".", 1)[0].replace("_", " ")
        ax.set_title(titulo, fontsize=10)

    plt.suptitle(f"{oracion}", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

    mostradas += 1

# ==========================
# RESUMEN FINAL
# ==========================
no_encontrados_set = list(dict.fromkeys(no_encontrados_global))  # únicos, preservando orden
print("\n==========================")
print("RESUMEN DE IMÁGENES FALTANTES")
print("==========================")
print(f"Total faltantes (únicas): {len(no_encontrados_set)}")
if no_encontrados_set:
    print("Ejemplos:")
    for ej in no_encontrados_set[:20]:
        print("  -", ej)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title SECUENCIAS CULTURA
import os
import json
import unicodedata
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ==========================
# RUTAS
# ==========================
JSON_CANDIDATOS = [
    "/content/selecciones_cultura_final.json",
    "/mnt/data/selecciones_cultura_final.json",
]

# carpeta con todos los pictogramas
PICTOS_DIR = "/content/todos_pictogramas"

# ==========================
# PARÁMETROS OPCIONALES
# ==========================
MAX_ORACIONES_A_MOSTRAR = None   # p.ej. 20 para limitar; None para todas
EXT_PERMITIDAS = {".jpg", ".jpeg", ".png"}

#   "secuencia_final"  -> 1 pictograma por token (la selección final)
#   "elecciones"       -> todas las imágenes de 'elecciones' (para revisar candidatos)
MODO_IMAGENES = "secuencia_final"

# ==========================
# UTILIDADES
# ==========================
def normalizar_txt(s: str) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = s.replace(" ", "_")
    return s

def extraer_id(nombre: str):
    """
    Si nombre es '7106-dolor_de_cabeza.jpg' devuelve '7106'.
    Si no hay guion, devuelve None.
    """
    base = os.path.basename(nombre)
    if "-" in base:
        return base.split("-", 1)[0]
    return None

def quitar_prefijo_orden(nombre: str):
    """
    Quita prefijos tipo '01_', '002_' SOLO si están al inicio del nombre.
    Ej.: '01_dolor_de_cabeza.jpg' -> 'dolor_de_cabeza.jpg'
         '7156-dolor_de_cabeza.jpg' -> (sin cambios)
    """
    import re, os
    base = os.path.basename(nombre or "")
    return re.sub(r'^\d{1,4}_', '', base)


def resolver_ruta_picto(nombre_entrada: str,
                        index_por_nombre, index_por_stem, index_por_id):
    import unicodedata, os

    def sin_tildes(s: str) -> str:
        if not s: return s
        nf = unicodedata.normalize("NFD", s)
        return "".join(ch for ch in nf if unicodedata.category(ch) != "Mn")

    if not nombre_entrada:
        return None

    candidatos_nombre = [os.path.basename(nombre_entrada)]
    nombre_sin_pref = quitar_prefijo_orden(nombre_entrada)
    if nombre_sin_pref != candidatos_nombre[0]:
        candidatos_nombre.append(nombre_sin_pref)

    for nombre in candidatos_nombre:
        base = os.path.basename(nombre)
        base_norm = normalizar_txt(base)
        stem_norm = normalizar_txt(os.path.splitext(base)[0])

        # 1) Búsqueda exacta por nombre+ext
        if base_norm in index_por_nombre:
            return index_por_nombre[base_norm]

        # 2) Probar otras extensiones si el stem coincide
        for ext in EXT_PERMITIDAS:
            cand = stem_norm + ext
            if cand in index_por_nombre:
                return index_por_nombre[cand]

        # 3) Por stem (sin extensión)
        if stem_norm in index_por_stem:
            return index_por_stem[stem_norm]

        # 3b) Fallback: sin tildes
        stem_noacc = sin_tildes(stem_norm)
        if stem_noacc != stem_norm:
            for ext in EXT_PERMITIDAS:
                cand = stem_noacc + ext
                if cand in index_por_nombre:
                    return index_por_nombre[cand]
            if stem_noacc in index_por_stem:
                return index_por_stem[stem_noacc]

        # 4) Por ID (parte antes del guion)
        _id = extraer_id(base)
        if _id and _id in index_por_id:
            return index_por_id[_id]

        # 5) Variantes guion/guion_bajo
        variantes = {
            base_norm.replace("-", "_"),
            base_norm.replace("_", "-"),
            stem_norm.replace("-", "_"),
            stem_norm.replace("_", "-"),
        }
        for var in variantes:
            # probar con extensión
            for ext in EXT_PERMITIDAS:
                cand = var.replace(ext, "") + ext
                if cand in index_por_nombre:
                    return index_por_nombre[cand]
            # probar sin extensión por stem
            if var in index_por_stem:
                return index_por_stem[var]

    return None

def elegir_json_path(candidatos):
    for p in candidatos:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "No se encontró el archivo JSON. "
        f"Probé: {', '.join(candidatos)}"
    )

# ==========================
# INDEXAR PICTOGRAMAS
# ==========================
print("Indexando pictogramas... (esto es rápido y solo se hace una vez)")

index_por_nombre = {}  # nombre exacto (normalizado) -> ruta
index_por_stem = {}    # nombre sin extensión (normalizado) -> ruta
index_por_id = {}      # ID numérica (texto) -> ruta (primera aparición)

total_archivos = 0
for root, _, files in os.walk(PICTOS_DIR):
    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext not in EXT_PERMITIDAS:
            continue
        total_archivos += 1
        ruta = os.path.join(root, fname)

        fname_norm = normalizar_txt(fname)
        stem_norm = normalizar_txt(os.path.splitext(fname)[0])

        index_por_nombre[fname_norm] = ruta
        if stem_norm not in index_por_stem:
            index_por_stem[stem_norm] = ruta

        _id = extraer_id(fname)
        if _id and _id.isdigit() and _id not in index_por_id:
            index_por_id[_id] = ruta

print(f"Archivos indexados: {total_archivos}")

# ==========================
# CARGAR JSON
# ==========================
JSON_PATH = elegir_json_path(JSON_CANDIDATOS)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} oraciones desde {JSON_PATH}.")

# Validar clave de imágenes elegida
if MODO_IMAGENES not in {"secuencia_final", "elecciones"}:
    raise ValueError("MODO_IMAGENES debe ser 'secuencia_final' o 'elecciones'.")

# ==========================
# MOSTRAR ORACIONES + PICTOGRAMAS
# ==========================
no_encontrados_global = []
mostradas = 0

for idx, item in enumerate(data, start=1):
    if MAX_ORACIONES_A_MOSTRAR is not None and mostradas >= MAX_ORACIONES_A_MOSTRAR:
        break

    oracion = item.get("oracion", "")
    lista_imgs = item.get(MODO_IMAGENES, []) or []
    # filtrar vacíos/null
    pictos = [p for p in lista_imgs if p]

    print(f"\n{idx}. {oracion}")
    if not pictos:
        print(f"   No hay imágenes en '{MODO_IMAGENES}' para esta oración.")
        continue

    imgs = []
    faltantes = []
    for fname in pictos:
        ruta = resolver_ruta_picto(fname, index_por_nombre, index_por_stem, index_por_id)
        if ruta and os.path.exists(ruta):
            try:
                img = mpimg.imread(ruta)
                imgs.append((os.path.basename(ruta), img))
            except Exception as e:
                print(f"   Error al leer {ruta}: {e}")
                faltantes.append(fname)
        else:
            faltantes.append(fname)

    if faltantes:
        for f in faltantes:
            print(f"   No se encontró la imagen: {f}")
        no_encontrados_global.extend(faltantes)

    if not imgs:
        print("   No se encontraron imágenes válidas para mostrar.")
        continue

    # Mostrar imágenes en fila
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]

    for ax, (fname_img, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        # Título debajo de cada imagen: parte después del '-' sin extensión
        titulo = fname_img
        if "-" in fname_img:
            titulo = fname_img.split("-", 1)[-1]
        titulo = titulo.rsplit(".", 1)[0].replace("_", " ")
        ax.set_title(titulo, fontsize=10)

    plt.suptitle(f"{oracion}", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

    mostradas += 1

# ==========================
# RESUMEN FINAL
# ==========================
no_encontrados_set = list(dict.fromkeys(no_encontrados_global))  # únicos, preservando orden
print("\n==========================")
print("RESUMEN DE IMÁGENES FALTANTES")
print("==========================")
print(f"Total faltantes (únicas): {len(no_encontrados_set)}")
if no_encontrados_set:
    print("Ejemplos:")
    for ej in no_encontrados_set[:20]:
        print("  -", ej)

Output hidden; open in https://colab.research.google.com to view.